# 10 — Moteur de décision
Politique : REFUS si score B ≥ seuil B (figé) ; REVUE si score B moyen OU score A2 élevé ; sinon APPROBATION.
Seuils de revue calibrés sur la validation (budget 0,2 % des transactions, moitié B / moitié A2),
figés dans config/decision.yaml, puis évaluation unique sur les tests (en nombre et en montant).

In [ ]:
import gc
import json
import sys
from pathlib import Path

import duckdb
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

pd.set_option("display.width", 250)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml import decision
from secumon.ml.models import anomaly, dataset

GOLD = ROOT / "data" / "processed" / "gold" / "gold.parquet"
MODELS_A = ROOT / "data" / "processed" / "models" / "engine_a"
MODELS_B = ROOT / "data" / "processed" / "models" / "engine_b"
POLICY_FILE = ROOT / "config" / "decision.yaml"

con = duckdb.connect()
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")
con.execute("SET enable_progress_bar = false")

meta_b = json.loads((MODELS_B / "meta.json").read_text())
meta_a2 = json.loads((MODELS_A / "meta_v2.json").read_text())
model_b = lgb.Booster(model_file=str(MODELS_B / "lgbm.txt"))
model_a2 = joblib.load(MODELS_A / "isolation_forest_v2.joblib")   # fichier produit par nous-memes
print(f"Seuil B fige : {meta_b['threshold']:.4f} | seuil A2 fige : {meta_a2['threshold']:.4f}")

In [ ]:
morceaux = []
for part in dataset.iter_split(con, GOLD, "validation", n_parts=4):
    morceaux.append(pd.DataFrame({
        "is_fraud": part["is_fraud"].to_numpy(dtype=bool),
        "score_b": model_b.predict(dataset.features_frame(part, meta_b["categories"])),
        "score_a": anomaly.anomaly_score(model_a2, anomaly.anomaly_matrix(part)),
    }))
    del part
    gc.collect()
val = pd.concat(morceaux, ignore_index=True)
del morceaux

print(f"Validation : {len(val):,} transactions, {int(val['is_fraud'].sum()):,} fraudes")
print(f"Controle B  : PR-AUC {average_precision_score(val['is_fraud'], val['score_b']):.4f} (attendu 0.8964)")
print(f"Controle A2 : PR-AUC {average_precision_score(val['is_fraud'], val['score_a']):.4f} (attendu 0.0677)")

In [ ]:
REVIEW_RATE, SHARE_B = 0.002, 0.5

policy = decision.calibrate_review_thresholds(
    val["score_b"], val["score_a"], decline_b=meta_b["threshold"],
    review_rate=REVIEW_RATE, share_b=SHARE_B)
actions = decision.decide_batch(val["score_b"], val["score_a"], policy)

print(policy)
for act in [decision.DECLINE, decision.REVIEW, decision.APPROVE]:
    sel = actions == act
    print(f"{act:8s} : {sel.mean():.3%} des transactions | fraudes : {int((sel & val['is_fraud']).sum()):,}")

decision.save_policy(policy, POLICY_FILE, meta={
    "review_rate": REVIEW_RATE, "share_b": SHARE_B, "calibrated_on": "validation 2015",
    "engine_b": "lightgbm v1", "engine_a": "isolation_forest v2",
})
print("Politique figee :", POLICY_FILE)
print(POLICY_FILE.read_text())
del val, actions
gc.collect()

In [ ]:
policy = decision.load_policy(POLICY_FILE)          # on evalue la politique FIGEE, relue du fichier
tests = (pd.read_parquet(MODELS_B / "scores_tests.parquet")
         .merge(pd.read_parquet(MODELS_A / "scores_tests_v2.parquet"), on="tx_id"))
montants = con.sql(f"""
    SELECT tx_id, amount FROM read_parquet('{GOLD.as_posix()}')
    WHERE split IN ('test_a', 'test_b')
""").df()
tests = tests.merge(montants, on="tx_id")
print(f"Tests : {len(tests):,} transactions (attendu 7,214,337)")

D, R, A = decision.DECLINE, decision.REVIEW, decision.APPROVE
b, a2 = tests["score"].to_numpy(), tests["score_a2"].to_numpy()
politiques = {
    "1. B seul (refus)": np.where(b >= policy.decline_b, D, A),
    "2. A2 ou B (refus)": np.where((b >= policy.decline_b) | (a2 >= meta_a2["threshold"]), D, A),
    "3. Moteur de decision": decision.decide_batch(b, a2, policy),
}

def bilan(masque, actions):
    f = tests["is_fraud"].to_numpy(dtype=bool)[masque]
    m = tests["amount"].to_numpy()[masque]
    act = actions[masque]
    return {
        "fraudes": int(f.sum()),
        "fraudes REFUS": int(((act == D) & f).sum()),
        "fraudes REVUE": int(((act == R) & f).sum()),
        "fraudes MANQUEES": int(((act == A) & f).sum()),
        "legitimes REFUS": int(((act == D) & ~f).sum()),
        "legitimes REVUE": int(((act == R) & ~f).sum()),
        "$ fraude refus": round(float(m[(act == D) & f].sum())),
        "$ fraude revue": round(float(m[(act == R) & f].sum())),
        "$ fraude manquee": round(float(m[(act == A) & f].sum())),
    }

jeux = {
    "test_a": (tests["split"] == "test_a").to_numpy(),
    "test_b": (tests["split"] == "test_b").to_numpy(),
    "test_b Italie": ((tests["split"] == "test_b") & (tests["zone"] == "Italie")).to_numpy(),
}
lignes = [{"jeu": j, "politique": p, **bilan(mk, act)}
          for j, mk in jeux.items() for p, act in politiques.items()]
print(pd.DataFrame(lignes).to_string(index=False))

## Conclusion — moteur de décision v1

Politique figée (config/decision.yaml), calibrée sur la validation 2015 :
REFUS si score B ≥ 0,7743 ; REVUE si score B ≥ 0,0962 ou score A2 ≥ 0,6995 ; sinon APPROBATION.
Budget de revue : 0,2 % des transactions (moitié B, moitié A2).

| Jeu | Politique | Fraudes traitées | Clients honnêtes refusés | $ fraude manqués |
|---|---|---|---|---|
| Test A | B seul | 73,6 % | 380 | 131 525 |
| Test A | A2 ou B (refus) | 75,1 % | 2 650 | 117 710 |
| Test A | **Moteur de décision** | **84,2 %** | **380** | **73 522** |
| Test B | B seul | 41,1 % | 1 024 | 199 507 |
| Test B | A2 ou B (refus) | 53,3 % | 9 826 | 157 903 |
| Test B | **Moteur de décision** | **67,2 %** | **1 024** | **102 467** |
| Test B Italie | **Moteur de décision** | **66,8 %** (B seul : 40,2 %) | 2 | 100 484 |

Le moteur de décision traite beaucoup plus de fraudes **sans refuser un seul client honnête de plus**
que B seul : le signal bruyant de A2 part en revue analyste au lieu d'un refus.
Le « OU » simple bloquait 7 à 10 fois plus de clients honnêtes.

**Limite** : une fraude en revue n'est pas empêchée d'office (transaction approuvée) ; son intérêt est
le blocage de la carte par l'analyste, qui empêche les fraudes suivantes (~9 par carte dans la
campagne) — effet à simuler.